conjuntos

In [3]:
archivo = "../../ejercicios_practicos/datos_matrimonio/marriage_longevity_master.csv"

In [5]:
import csv
from pathlib import Path

RUTA = Path(archivo)

def cargar():
    """Lee el CSV y devuelve una lista de diccionarios (uno por matrimonio)."""
    with open(RUTA, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def conjunto_ids(filas, predicado):
    """Devuelve un SET de marriage_id que cumplen la condición.
    El set elimina duplicados (propiedad de los conjuntos)."""
    return {f["marriage_id"] for f in filas if predicado(f)}

filas = cargar()

# Definimos conjuntos (sucesos) sobre el espacio muestral de matrimonios
universo = {f["marriage_id"] for f in filas}          # Ω : todos los matrimonios

casados = conjunto_ids(filas, lambda f: f["divorced"] == "0")       # sigue casado
divorciados = conjunto_ids(filas, lambda f: f["divorced"] == "1")   # se divorció
licenciados = conjunto_ids(filas, lambda f: f["education_level"] == "bachelors")
terapia = conjunto_ids(filas, lambda f: f["premarital_counseling"] == "1")

print(f"|Ω| (universo)                 = {len(universo)}")
print(f"|casados|                      = {len(casados)}")
print(f"|divorciados|                  = {len(divorciados)}")

# ---- OPERACIONES DE CONJUNTOS (y su lectura probabilística) ----
# Unión: casados O divorciados  →  debe cubrir el universo
union = casados | divorciados
print(f"\nA ∪ B (casados ∪ divorciados)  = {len(union)}  "
      f"¿cubre el universo? {union == universo}")

# Intersección: casados Y divorciados  →  mutuamente excluyentes?
inter = casados & divorciados
print(f"A ∩ B (casados ∩ divorciados)  = {len(inter)}  "
      f"¿disjuntos? {inter == set()}")

# Diferencia: licenciados que NO están divorciados
licen_sin_divorcio = licenciados - divorciados
print(f"licenciados - divorciados      = {len(licen_sin_divorcio)}")

# Complemento: matrimonios no licenciados  →  |A^c| = |Ω| - |A|
complemento_lic = universo - licenciados
print(f"A^c (no licenciados)           = {len(complemento_lic)}  "
      f"|Ω|-|A| = {len(universo) - len(licenciados)}")

# Pertenencia: primer matrimonio, ¿es bachelors?
ejemplo_id = next(iter(divorciados))
print(f"\nPertenencia: ¿{ejemplo_id} ∈ licenciados? "
      f"{ejemplo_id in licenciados}")
print(f"Pertenencia: ¿{ejemplo_id} ∈ divorciados? "
      f"{ejemplo_id in divorciados}")

# Sets no repiten elementos: misma propiedad da el mismo cardinal
print(f"\n|licenciados| con set = {len(licenciados)}")

|Ω| (universo)                 = 45000
|casados|                      = 24292
|divorciados|                  = 20708

A ∪ B (casados ∪ divorciados)  = 45000  ¿cubre el universo? True
A ∩ B (casados ∩ divorciados)  = 0  ¿disjuntos? True
licenciados - divorciados      = 7199
A^c (no licenciados)           = 33300  |Ω|-|A| = 33300

Pertenencia: ¿MAR031230 ∈ licenciados? False
Pertenencia: ¿MAR031230 ∈ divorciados? True

|licenciados| con set = 11700


sub conjuntos

In [6]:
import csv
from pathlib import Path

RUTA = Path(archivo)

def cargar():
    # Lee el CSV y devuelve una lista de diccionarios (uno por matrimonio)
    with open(RUTA, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def conjunto_ids(filas, predicado):
    # Devuelve un SET de marriage_id que cumplen la condición
    return {f["marriage_id"] for f in filas if predicado(f)}

filas = cargar()

# ---------- Definición de conjuntos ----------
universo = {f["marriage_id"] for f in filas}                    # Ω : todos
casados = conjunto_ids(filas, lambda f: f["divorced"] == "0")
divorciados = conjunto_ids(filas, lambda f: f["divorced"] == "1")

# A: nivel educativo bachelors
A_licenciados = conjunto_ids(filas, lambda f: f["education_level"] == "bachelors")

# B: bachelors Y con terapia prematrimonial  →  B es subconjunto de A
B_licen_terapia = conjunto_ids(
    filas,
    lambda f: f["education_level"] == "bachelors"
              and f["premarital_counseling"] == "1",
)

print(f"|Ω| (universo)             = {len(universo)}")
print(f"|A| (bachelors)            = {len(A_licenciados)}")
print(f"|B| (bachelors + terapia)  = {len(B_licen_terapia)}")

# ---------- Verificación de subconjuntos (issubset / issuperset) ----------
print(f"\nB ⊆ A  →  {B_licen_terapia.issubset(A_licenciados)}")
print(f"A ⊆ Ω  →  {A_licenciados.issubset(universo)}")
print(f"Ω ⊇ A  →  {universo.issuperset(A_licenciados)}")

# ---------- Propiedades de la contención ----------
print(f"\nReflexiva:  A ⊆ A          →  {A_licenciados.issubset(A_licenciados)}")
print(f"Antisimetría: A⊆B y B⊆A ⟹ A==B →  {A_licenciados == A_licenciados}")

transitiva = (B_licen_terapia.issubset(A_licenciados)
              and A_licenciados.issubset(universo)
              and B_licen_terapia.issubset(universo))
print(f"Transitiva: B⊆A, A⊆Ω ⟹ B⊆Ω  →  {transitiva}")

# ---------- No subconjunto ----------
print(f"\n¿casados ⊆ divorciados?    →  {casados.issubset(divorciados)}")
print(f"¿∅ ⊆ A?                    →  {set().issubset(A_licenciados)}")

# ---------- Probabilidad condicional desde subconjuntos ----------
# Como B ⊆ A, la probabilidad de B dado A es |B| / |A|
p = len(B_licen_terapia) / len(A_licenciados)
print(f"\nP(B|A) = |B|/|A| = {len(B_licen_terapia)}/{len(A_licenciados)} = {p:.4f}")
print(f"P(B) = |B|/|Ω| = {len(B_licen_terapia)}/{len(universo)} = {len(B_licen_terapia)/len(universo):.4f}")


|Ω| (universo)             = 45000
|A| (bachelors)            = 11700
|B| (bachelors + terapia)  = 2987

B ⊆ A  →  True
A ⊆ Ω  →  True
Ω ⊇ A  →  True

Reflexiva:  A ⊆ A          →  True
Antisimetría: A⊆B y B⊆A ⟹ A==B →  True
Transitiva: B⊆A, A⊆Ω ⟹ B⊆Ω  →  True

¿casados ⊆ divorciados?    →  False
¿∅ ⊆ A?                    →  True

P(B|A) = |B|/|A| = 2987/11700 = 0.2553
P(B) = |B|/|Ω| = 2987/45000 = 0.0664


Diferencia de conjuntos

In [4]:
import csv
from pathlib import Path

RUTA = Path(archivo)

def cargar():
    # Lee el CSV y devuelve una lista de diccionarios (uno por matrimonio)
    with open(RUTA, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def conjunto_ids(filas, predicado):
    # Devuelve un SET de marriage_id que cumplen la condición
    return {f["marriage_id"] for f in filas if predicado(f)}

filas = cargar()
A = conjunto_ids(filas, lambda f: f["education_level"] == "bachelors")   # bachelors
T = conjunto_ids(filas, lambda f: f["premarital_counseling"] == "1")     # terapia
D = conjunto_ids(filas, lambda f: f["divorced"] == "1")                  # divorciados

print(f"|A| = {len(A)}   |T| = {len(T)}   |D| = {len(D)}")

# ---------- Diferencia: quitar lo compartido ----------
print(f"\n|A - T| (bachelors sin terapia)   = {len(A - T)}")
print(f"|A - A| (conjunto consigo mismo)   = {len(A - A)}")

# ---------- No conmutativa ----------
print(f"\n|A - T| = {len(A - T)}   |T - A| = {len(T - A)}")
print(f"¿A - T == T - A?   →  {(A - T) == (T - A)}")

# ---------- Equivalencia con A ∩ B^c ----------
comp_T = {f["marriage_id"] for f in filas} - T      # Ω - T (no hicieron terapia)
print(f"\n¿A - T == A ∩ T^c?     →  {(A - T) == (A & comp_T)}")

# ---------- Partición de A ----------
print(f"\n|(A - T) ∪ (A ∩ T)|  = {len((A - T) | (A & T))}  (debe ser |A| = {len(A)})")
print(f"¿(A - T) ∩ (A ∩ T) = ∅?  →  {(A - T) & (A & T) == set()}")

# ---------- Diferencia con universo y disjuntos ----------
Omega = {f["marriage_id"] for f in filas}
print(f"\n|Ø| (A - A)          = {len(A - A)}")
print(f"|A - Ω|             = {len(A - Omega)}  (disjuntos e intersección: nada que quitar)")
print(f"|Ω - D|             = {len(Omega - D)}  (= |casados|)")



|A| = 11700   |T| = 10972   |D| = 20708

|A - T| (bachelors sin terapia)   = 8713
|A - A| (conjunto consigo mismo)   = 0

|A - T| = 8713   |T - A| = 7985
¿A - T == T - A?   →  False

¿A - T == A ∩ T^c?     →  True

|(A - T) ∪ (A ∩ T)|  = 11700  (debe ser |A| = 11700)
¿(A - T) ∩ (A ∩ T) = ∅?  →  True

|Ø| (A - A)          = 0
|A - Ω|             = 0  (disjuntos e intersección: nada que quitar)
|Ω - D|             = 24292  (= |casados|)
